In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from collections import Counter
import re

In [3]:
# Load Dataset
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

# Convert labels
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Optional: reduce size for faster training
df = df.sample(30000, random_state=42)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [4]:
df.head()

,review,sentiment
33553,I really liked this Summerslam due to the look...,1
9427,Not many television shows appeal to quite as m...,1
199,The film quickly gets to a major chase scene w...,0
12447,Jane Austen would definitely approve of this o...,1
39489,Expectations were somewhat high for me when I ...,0


In [5]:
# Text Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

train_df['review'] = train_df['review'].apply(clean_text)
test_df['review'] = test_df['review'].apply(clean_text)

In [6]:
# Build Vocabulary
def tokenize(text):
    return text.split()

counter = Counter()
for text in train_df['review']:
    counter.update(tokenize(text))

vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(10000))}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

In [7]:
# Encode Text
max_len = 200

def encode(text):
    tokens = tokenize(text)
    seq = [vocab.get(word, 1) for word in tokens]
    seq = seq[:max_len] + [0]*(max_len - len(seq))
    return seq

X_train = np.array([encode(text) for text in train_df['review']])
y_train = np.array(train_df['sentiment'])

X_test = np.array([encode(text) for text in test_df['review']])
y_test = np.array(test_df['sentiment'])

In [8]:
# Convert to PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

In [9]:
# Model (RNN / LSTM / GRU)
class RNNModel(nn.Module):
    def __init__(self, rnn_type="RNN", vocab_size=10000, embed_dim=128, hidden_dim=128):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        if rnn_type == "RNN":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == "LSTM":
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == "GRU":
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        output, hidden = self.rnn(x)
        
        if isinstance(hidden, tuple):  # LSTM
            hidden = hidden[0]
        
        out = hidden[-1]
        return self.fc(out).squeeze()

In [10]:
# Training Function
def train_model(model, loader, epochs=8):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    
    for epoch in range(epochs):
        total_loss, correct = 0, 0
        
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            
            optimizer.zero_grad()
            outputs = model(X)
            
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == y).sum().item()
        
        acc = correct / len(loader.dataset)
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Acc: {acc:.4f}")

In [11]:
# Evaluation
def evaluate(model, loader):
    model.eval()
    correct = 0
    
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == y).sum().item()
    
    acc = correct / len(loader.dataset)
    print(f"Test Accuracy: {acc:.4f}")

In [12]:
# Train All Models
# RNN
rnn_model = RNNModel("RNN", vocab_size=len(vocab)).to(device)
train_model(rnn_model, train_loader)
evaluate(rnn_model, test_loader)

# LSTM
lstm_model = RNNModel("LSTM", vocab_size=len(vocab)).to(device)
train_model(lstm_model, train_loader)
evaluate(lstm_model, test_loader)

# GRU
gru_model = RNNModel("GRU", vocab_size=len(vocab)).to(device)
train_model(gru_model, train_loader)
evaluate(gru_model, test_loader)

Epoch 1, Loss: 522.4243, Acc: 0.5079
Epoch 2, Loss: 517.7426, Acc: 0.5260
Epoch 3, Loss: 514.7605, Acc: 0.5325
Epoch 4, Loss: 506.1644, Acc: 0.5572
Epoch 5, Loss: 494.6366, Acc: 0.5706
Epoch 6, Loss: 471.4785, Acc: 0.5927
Epoch 7, Loss: 459.5577, Acc: 0.6090
Epoch 8, Loss: 452.9777, Acc: 0.6171
Test Accuracy: 0.5143
Epoch 1, Loss: 519.3202, Acc: 0.5129
Epoch 2, Loss: 507.7895, Acc: 0.5622
Epoch 3, Loss: 460.6520, Acc: 0.6708
Epoch 4, Loss: 474.0490, Acc: 0.6368
Epoch 5, Loss: 312.2236, Acc: 0.8192
Epoch 6, Loss: 220.9736, Acc: 0.8830
Epoch 7, Loss: 159.7375, Acc: 0.9206
Epoch 8, Loss: 111.4878, Acc: 0.9513
Test Accuracy: 0.8345
Epoch 1, Loss: 519.2283, Acc: 0.5230
Epoch 2, Loss: 504.4539, Acc: 0.5728
Epoch 3, Loss: 278.2699, Acc: 0.8380
Epoch 4, Loss: 177.2509, Acc: 0.9056
Epoch 5, Loss: 117.3238, Acc: 0.9443
Epoch 6, Loss: 66.8223, Acc: 0.9712
Epoch 7, Loss: 38.7313, Acc: 0.9848
Epoch 8, Loss: 21.9613, Acc: 0.9913
Test Accuracy: 0.8533


In [13]:
def predict_text(model, text):
    model.eval()
    
    # 1. Clean text
    text = clean_text(text)
    
    # 2. Tokenize
    tokens = text.split()
    
    # 3. Encode
    seq = [vocab.get(word, 1) for word in tokens]
    
    # 4. Pad
    seq = seq[:max_len] + [0]*(max_len - len(seq))
    
    # 5. Convert to tensor
    seq = torch.tensor(seq, dtype=torch.long).unsqueeze(0).to(device)
    
    # 6. Predict
    with torch.no_grad():
        output = model(seq)
        prob = torch.sigmoid(output).item()
    
    # 7. Result
    if prob > 0.5:
        print(f"Positive 😊 ({prob:.4f})")
    else:
        print(f"Negative 😞 ({prob:.4f})")

In [14]:
predict_text(lstm_model, "This movie was amazing and fantastic")
predict_text(lstm_model, "Worst movie ever, totally waste of time")

Positive 😊 (0.9874)
Negative 😞 (0.0056)
